# Toponym Recognition

## Goal

Extract candidate place names from text with public NER tools.

## What you will do

- Learn what each NER tool is useful for.
- Install a tool only when you want to run that section.
- Run small examples with spaCy, Stanza, Flair, or Transformers.
- Compare outputs and save mentions to `outputs/results/ner_mentions.csv`.

In [3]:
from pathlib import Path
import importlib
import importlib.util
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import RESULTS_DIR
from src.data_utils import load_sample_texts, save_dataframe
import src.ner_utils as ner_utils

TRANSFORMERS_VERSION = "transformers>=4.25,<4.58"
EMPTY_MENTIONS = ["mention", "start", "end", "label", "tool", "source_text_id", "sentence"]


def refresh_imports(*package_names):
    for package_name in package_names:
        for module_name in list(sys.modules):
            if module_name == package_name or module_name.startswith(package_name + "."):
                del sys.modules[module_name]
    importlib.invalidate_caches()


def reload_ner_helpers():
    importlib.invalidate_caches()
    importlib.reload(ner_utils)
    globals().update(
        {
            "extract_locations_spacy": ner_utils.extract_locations_spacy,
            "extract_locations_stanza": ner_utils.extract_locations_stanza,
            "extract_locations_flair": ner_utils.extract_locations_flair,
            "extract_locations_transformers": ner_utils.extract_locations_transformers,
            "normalize_ner_results": ner_utils.normalize_ner_results,
            "combine_and_deduplicate_mentions": ner_utils.combine_and_deduplicate_mentions,
            "LOCATION_LABELS": ner_utils.LOCATION_LABELS,
        }
    )


def show_package_status(package_name, label):
    importlib.invalidate_caches()
    available = importlib.util.find_spec(package_name) is not None
    print(f"{label}: {'available' if available else 'missing'}")
    return available


def install_packages_if_missing(packages, module_names):
    missing = [name for name in module_names if importlib.util.find_spec(name) is None]
    if missing:
        print("Installing missing packages:", ", ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
    else:
        print("Required Python packages are already installed.")


def run_helper_in_subprocess(function_name, text, model_name, allow_download, timeout=180):
    code = "\n".join(
        [
            "import json, sys",
            f"sys.path.insert(0, {str(PROJECT_ROOT)!r})",
            f"from src.ner_utils import {function_name}",
            f"rows = {function_name}({text!r}, model_name={model_name!r}, allow_download={allow_download!r})",
            "print(json.dumps(rows, ensure_ascii=False))",
        ]
    )
    try:
        completed = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"{function_name} failed: timed out")
        return []

    output = completed.stdout.strip().splitlines()
    json_line = output[-1] if output else "[]"
    if completed.returncode != 0:
        message = (completed.stderr or completed.stdout).strip().splitlines()
        print(f"{function_name} failed:", message[-1] if message else f"exit code {completed.returncode}")
        return []
    try:
        return json.loads(json_line)
    except json.JSONDecodeError:
        print(f"{function_name} failed: could not read subprocess output")
        return []


def model_smoke_test(label, function_name, model_name, allow_download):
    text = "Paris and Berlin are often mentioned in European news."
    if function_name == "extract_locations_transformers":
        text = "Flooding was reported in Valencia and Barcelona."

    rows = run_helper_in_subprocess(function_name, text, model_name, allow_download=False)
    if rows:
        print(f"{label} model check: OK")
        return True

    if allow_download:
        print(f"{label} model is not available locally. Trying to download it now.")
        rows = run_helper_in_subprocess(function_name, text, model_name, allow_download=True)
        if rows:
            print(f"{label} model check: OK")
            return True

    print(f"{label} model check failed.")
    return False


def check_flair_runtime():
    if not show_package_status("flair", "Flair"):
        return False
    return show_package_status("transformers", "Transformers")


def check_transformers_runtime():
    ok = show_package_status("transformers", "Transformers")
    ok = show_package_status("torch", "PyTorch") and ok
    return ok


reload_ner_helpers()

## Step 1: NER labels

NER tools use different labels. This project treats `LOC`, `LOCATION`, `GPE`, `FAC`, `FACILITY`, and `NEL` as location-like labels.

`GPE` usually means countries, cities, or states. `LOC` usually means natural or general locations. `FAC` can mean buildings, airports, bridges, or other facilities.

## Step 2: Load example texts

In [4]:
texts = load_sample_texts()
all_mentions = []
texts

,text_id,text
0,sample_1,Paris and Berlin are often mentioned in Europe...
1,sample_2,Apple opened a new office in California.
2,sample_3,"Nach dem Hochwasser wurden Schäden in Bayern, ..."
3,sample_4,The earthquake affected Izmir and nearby villa...
4,sample_5,Several hydrology studies focus on the Rhine b...


## Step 3: spaCy

spaCy is the recommended first tool for the workshop: it is lightweight, fast, and easy to explain. Run the install cell once if spaCy or the small models are missing.

In [3]:
RUN_SPACY_INSTALL = False

if RUN_SPACY_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "de_core_news_sm"])
else:
    print("Skipped spaCy install. Set RUN_SPACY_INSTALL = True to install spaCy and the small English/German models.")

reload_ner_helpers()
show_package_status("spacy", "spaCy")

Skipped spaCy install. Set RUN_SPACY_INSTALL = True to install spaCy and the small English/German models.
spaCy: available


True

In [4]:
rows = []
for _, row in texts.iterrows():
    results = extract_locations_spacy(row["text"], model_name="en_core_web_sm")
    rows.append(normalize_ner_results(results, source_text_id=row["text_id"]))

spacy_mentions = combine_and_deduplicate_mentions(rows)
all_mentions.append(spacy_mentions)
spacy_mentions

,mention,start,end,label,tool,source_text_id,sentence
0,Paris,0,5,GPE,spacy,sample_1,Paris and Berlin are often mentioned in Europe...
1,Berlin,10,16,GPE,spacy,sample_1,Paris and Berlin are often mentioned in Europe...
2,California,29,39,GPE,spacy,sample_2,Apple opened a new office in California.
3,Bayern,38,44,GPE,spacy,sample_3,"Nach dem Hochwasser wurden Schäden in Bayern, ..."


In [5]:
german_text = texts.loc[texts["text_id"] == "sample_3", "text"].iloc[0]
german_spacy_mentions = normalize_ner_results(
    extract_locations_spacy(german_text, model_name="de_core_news_sm"),
    source_text_id="sample_3_de_spacy",
)
german_spacy_mentions

,mention,start,end,label,tool,source_text_id,sentence
0,Bayern,38,44,LOC,spacy,sample_3_de_spacy,"Nach dem Hochwasser wurden Schäden in Bayern, ..."
1,Passau,46,52,LOC,spacy,sample_3_de_spacy,"Nach dem Hochwasser wurden Schäden in Bayern, ..."
2,Österreich,57,67,LOC,spacy,sample_3_de_spacy,"Nach dem Hochwasser wurden Schäden in Bayern, ..."


## Step 4: Stanza

Stanza is useful when you want a Stanford NLP pipeline and broad language coverage. Its language models are downloaded separately.

In [ ]:
RUN_STANZA_INSTALL = True

if RUN_STANZA_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "stanza"])
    import stanza
    stanza.download("en")
else:
    print("Skipped Stanza install. Set RUN_STANZA_INSTALL = True to install Stanza and download the English model.")

reload_ner_helpers()
show_package_status("stanza", "Stanza")

In [5]:
stanza_results = extract_locations_stanza("The earthquake affected Izmir and nearby villages.", lang="en")
stanza_mentions = normalize_ner_results(stanza_results, source_text_id="stanza_demo")
all_mentions.append(stanza_mentions)
stanza_mentions

,mention,start,end,label,tool,source_text_id,sentence
0,Izmir,24,29,GPE,stanza,stanza_demo,The earthquake affected Izmir and nearby villa...


## Step 5: Flair

Flair provides strong sequence labeling models. It is heavier than spaCy, so use it when you want to compare model behavior.

In [7]:
RUN_FLAIR_INSTALL = True
FLAIR_MODEL_NAME = "flair/ner-english"

if RUN_FLAIR_INSTALL:
    install_packages_if_missing(["flair", TRANSFORMERS_VERSION], ["flair", "transformers"])
else:
    print("Skipped Flair setup. Set RUN_FLAIR_INSTALL = True to install Flair and prepare the NER model.")

refresh_imports("flair", "transformers")
reload_ner_helpers()
FLAIR_READY = check_flair_runtime()
FLAIR_MODEL_READY = FLAIR_READY and model_smoke_test(
    "Flair",
    "extract_locations_flair",
    FLAIR_MODEL_NAME,
    allow_download=RUN_FLAIR_INSTALL,
)

if not FLAIR_MODEL_READY:
    print("Flair is not ready. Check internet/model access, then rerun this Flair setup cell.")

Required Python packages are already installed.
Flair: available
Transformers: available
Flair model is not available locally. Trying to download it now.
Flair model check: OK


In [10]:
all_mentions = globals().setdefault("all_mentions", [])

if globals().get("FLAIR_MODEL_READY", False):
    flair_results = run_helper_in_subprocess(
        "extract_locations_flair",
        "Paris and Berlin are often mentioned in European news.",
        FLAIR_MODEL_NAME,
        allow_download=False,
    )
else:
    print("Flair example skipped because Flair setup did not complete.")
    flair_results = []

flair_mentions = normalize_ner_results(flair_results, source_text_id="flair_demo")
all_mentions.append(flair_mentions)
flair_mentions

,mention,start,end,label,tool,source_text_id,sentence
0,Paris,0,5,LOC,flair,flair_demo,Paris and Berlin are often mentioned in Europe...
1,Berlin,10,16,LOC,flair,flair_demo,Paris and Berlin are often mentioned in Europe...


## Step 6: Transformers

Transformers pipelines are useful for comparing neural multilingual NER models. They are heavier than spaCy and usually need model weights from the local Hugging Face cache or internet.

In [11]:
RUN_TRANSFORMERS_INSTALL = True
TRANSFORMERS_MODEL_NAME = "Davlan/xlm-roberta-base-ner-hrl"

if RUN_TRANSFORMERS_INSTALL:
    install_packages_if_missing([TRANSFORMERS_VERSION, "torch"], ["transformers", "torch"])
else:
    print("Skipped Transformers setup. Set RUN_TRANSFORMERS_INSTALL = True to install Transformers/PyTorch and prepare the NER model.")

refresh_imports("transformers", "torch")
reload_ner_helpers()
TRANSFORMERS_READY = check_transformers_runtime()
TRANSFORMERS_MODEL_READY = TRANSFORMERS_READY and model_smoke_test(
    "Transformers",
    "extract_locations_transformers",
    TRANSFORMERS_MODEL_NAME,
    allow_download=RUN_TRANSFORMERS_INSTALL,
)

if not TRANSFORMERS_MODEL_READY:
    print("Transformers is not ready. Check internet/model access, then rerun this Transformers setup cell.")

Required Python packages are already installed.
Transformers: available
PyTorch: available
Transformers model check: OK


In [12]:
all_mentions = globals().setdefault("all_mentions", [])

if globals().get("TRANSFORMERS_MODEL_READY", False):
    transformer_results = run_helper_in_subprocess(
        "extract_locations_transformers",
        "Flooding was reported in Valencia and Barcelona.",
        TRANSFORMERS_MODEL_NAME,
        allow_download=False,
    )
else:
    print("Transformers example skipped because Transformers setup did not complete.")
    transformer_results = []

transformer_mentions = normalize_ner_results(transformer_results, source_text_id="transformers_demo")
all_mentions.append(transformer_mentions)
transformer_mentions

,mention,start,end,label,tool,source_text_id,sentence
0,Valencia,24,33,LOC,transformers,transformers_demo,Flooding was reported in Valencia and Barcelona.
1,Barcelona,37,47,LOC,transformers,transformers_demo,Flooding was reported in Valencia and Barcelona.


## Step 7: Compare and deduplicate mentions

In [13]:
combined = combine_and_deduplicate_mentions(all_mentions)
combined

,mention,start,end,label,tool,source_text_id,sentence
0,Paris,0,5,GPE,spacy,sample_1,Paris and Berlin are often mentioned in Europe...
1,Berlin,10,16,GPE,spacy,sample_1,Paris and Berlin are often mentioned in Europe...
2,California,29,39,GPE,spacy,sample_2,Apple opened a new office in California.
3,Bayern,38,44,GPE,spacy,sample_3,"Nach dem Hochwasser wurden Schäden in Bayern, ..."
4,Izmir,24,29,GPE,stanza,stanza_demo,The earthquake affected Izmir and nearby villa...
5,Paris,0,5,LOC,flair,flair_demo,Paris and Berlin are often mentioned in Europe...
6,Berlin,10,16,LOC,flair,flair_demo,Paris and Berlin are often mentioned in Europe...
7,Valencia,24,33,LOC,transformers,transformers_demo,Flooding was reported in Valencia and Barcelona.
8,Barcelona,37,47,LOC,transformers,transformers_demo,Flooding was reported in Valencia and Barcelona.


## Step 8: Save mentions

NER only extracts candidate place names. It does not resolve them to coordinates.

In [14]:
out = save_dataframe(combined, RESULTS_DIR / 'ner_mentions.csv')
print('Saved:', out)

Saved: /home/hu_xk/Workplace/wawopensearch3_hackathon/module2_geoparsing/outputs/results/ner_mentions.csv


## Exercise

Add your own text and run it with at least two tools. Compare which place names are found and which labels they receive.

In [6]:
my_text = "I travelled from Munich to Zurich and then to Vienna."
my_spacy = normalize_ner_results(extract_locations_spacy(my_text), source_text_id="my_text_spacy")
my_stanza = normalize_ner_results(extract_locations_stanza(my_text), source_text_id="my_text_stanza")
combine_and_deduplicate_mentions([my_spacy, my_stanza])

,mention,start,end,label,tool,source_text_id,sentence
0,Munich,17,23,GPE,spacy,my_text_spacy,I travelled from Munich to Zurich and then to ...
1,Zurich,27,33,GPE,spacy,my_text_spacy,I travelled from Munich to Zurich and then to ...
2,Vienna,46,52,GPE,spacy,my_text_spacy,I travelled from Munich to Zurich and then to ...
3,Munich,17,23,GPE,stanza,my_text_stanza,I travelled from Munich to Zurich and then to ...
4,Zurich,27,33,GPE,stanza,my_text_stanza,I travelled from Munich to Zurich and then to ...
5,Vienna,46,52,GPE,stanza,my_text_stanza,I travelled from Munich to Zurich and then to ...


## Common issues

- If a package or model is missing, run the setup cell in that tool's section.
- If spaCy is installed but the model is missing, rerun the spaCy setup cell.
- Stanza, Flair, and Transformers use model weights separately from the Python packages.
- Flair and Transformers setup cells now test the exact model used by the example before the example cell runs.
- Some `FAC` entities are useful locations; others are not, depending on the application.